## Use this notebook to run unit tests for features and functionality

In [1]:
#from filters import process_user_query, keep_price_and_bedroom_filters, build_filter_extraction_prompt, validate_filters
from vectorstore import hybrid_search, hybrid_search_raw, get_index
from chatdb import get_connection
from rag_pipeline import rag_search
import json
from pprint import pprint

In [2]:
conn = get_connection()
cur = conn.cursor()

In [3]:
# Get unique amenities
cur.execute("""
SELECT DISTINCT unnest(amenities) as amenity 
FROM listings 
WHERE amenities IS NOT NULL AND array_length(amenities, 1) > 0
ORDER BY amenity
""")
amenities = [row[0] for row in cur.fetchall() if row[0]]

# Get unique neighborhoods
cur.execute("""
SELECT DISTINCT neighborhood 
FROM listings 
WHERE neighborhood IS NOT NULL
ORDER BY neighborhood
""")
neighborhoods = [row[0] for row in cur.fetchall() if row[0]]

# Get unique subway routes
cur.execute("""
SELECT DISTINCT unnest(routes) as route 
FROM subway_stations 
WHERE routes IS NOT NULL
ORDER BY route
""")
routes = [row[0] for row in cur.fetchall() if row[0]]

# Get unique subway lines
cur.execute("""
SELECT DISTINCT line 
FROM subway_stations 
WHERE line IS NOT NULL
ORDER BY line
""")
lines = [row[0] for row in cur.fetchall() if row[0]]

In [4]:
result = {
    'amenities': amenities,
    'neighborhoods': neighborhoods,
    'subway_routes': routes,
    'subway_lines': lines
}

print(json.dumps(result, indent=2))

cur.close()
conn.close()

{
  "amenities": [
    "assigned_parking",
    "balcony",
    "basement_finished",
    "basement_full",
    "basement_partial",
    "bike_room",
    "cats",
    "central_ac",
    "childrens_playroom",
    "city_view",
    "co_purchase",
    "cold_storage",
    "concierge",
    "courtyard",
    "deck",
    "decorative_fireplace",
    "dishwasher",
    "dogs",
    "doorman",
    "elevator",
    "fios_available",
    "fireplace",
    "full_time_doorman",
    "furnished",
    "garage",
    "garage_attached",
    "garden",
    "garden_view",
    "gas_fireplace",
    "gifts",
    "guarantors",
    "gym",
    "hardwood_floors",
    "hot_tub",
    "land_lease",
    "laundry",
    "leed_registered",
    "live_in_super",
    "locker_cage",
    "media_room",
    "nyc_evacuation_1",
    "nyc_evacuation_2",
    "nyc_evacuation_3",
    "nyc_evacuation_4",
    "nyc_evacuation_5",
    "nyc_evacuation_6",
    "package_room",
    "parents",
    "park_view",
    "parking",
    "part_time_doorman",
    "p

In [5]:
# rag_pipeline.py
import os
from openai import OpenAI
from vectorstore import hybrid_search, parallel_hybrid_search
from rewriter import rewrite_query
from pinecone_filters import extract_pinecone_filters, build_pinecone_filter_prompt, _pydantic_to_pinecone_filters, PineconeFiltersResponse
from post_filters import (apply_post_retrieval_filters, parse_amenities, parse_neighborhoods, parse_subway_preferences,
                          build_pinecone_filter, combine_soft_with_hard)
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from rate_limiter import call_llm_with_limit
import asyncio
from response_router import decide_response_type
from filters_change import have_filters_changed
from scorer import score_listings
from rag_pipeline import _detect_new_search_intent, format_listings
from google import genai
from google.genai import types
from utils import deduplicate_matches

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [6]:
# Enter user's query
user_query = "Looking for a 2 bedroom apartment under 5000, in fidi with a gym and close to subway lines A or C."

In [7]:
# Other arguments
top_k=5
chat_history=None
is_first_turn: bool = False
previous_filters: dict = None
previous_matches=None

In [8]:
# Step -1: Detect if user wants a new search (overrides is_first_turn for formatting)
is_new_search = is_first_turn or _detect_new_search_intent(user_query, chat_history)
print(f"Is new search: {is_new_search}")

Is new search: True


In [9]:
# Step 0: Rewrite follow-up into standalone query
standalone_query = rewrite_query(user_query, chat_history or [])

In [10]:
# Step 1: Extract simple Pinecone pre-filters (price, bed, bath, sqft, zipcode ONLY)
pinecone_filters = await extract_pinecone_filters(standalone_query)

In [11]:
print("Extracted Pinecone Filters:", pinecone_filters)

Extracted Pinecone Filters: {'bedrooms': {'$eq': 2}, 'price': {'$lt': 5000.0}}


In [12]:
# Step 1.5: Decide response type (general vs index_query)
response_type = decide_response_type(user_query)

In [13]:
# Step 1.6: Determine if filters changed relative to previous
filters_changed = have_filters_changed(previous_filters or {}, pinecone_filters or {})

In [14]:
# Step 2: Parse post-retrieval filter criteria (neighborhoods, amenities, subway) concurrently
try:
    amenities, neighborhoods, subway_prefs = await asyncio.gather(
        parse_amenities(standalone_query),
        parse_neighborhoods(standalone_query),
        parse_subway_preferences(standalone_query)
    )

    # Defaults if any come back as None
    amenities = amenities or []
    neighborhoods = neighborhoods or []
    subway_prefs = subway_prefs or {}

except Exception as e:
    print(f"[WARN] Post-filter parsing error: {e}")
    # Fallback defaults
    amenities = []
    neighborhoods = []
    subway_prefs = {}

In [15]:
# Adaptive over-fetch: if we have both price and bedrooms, fetch less
has_strong_filters = bool(pinecone_filters.get("price")) and bool(pinecone_filters.get("bedrooms"))
print(f"Has strong filters: {has_strong_filters}")
retrieval_k = top_k * (2 if has_strong_filters else 3)
print(f"Retrieval k: {retrieval_k}")

Has strong filters: True
Retrieval k: 10


In [16]:
# Compile soft filters 
soft_filters = build_pinecone_filter(amenities, neighborhoods, subway_prefs)
print("Soft Filters:", soft_filters)

# Combine soft and hard filters
filter_list = combine_soft_with_hard(soft_filters, pinecone_filters)
print("Combined Filters:", filter_list)

Soft Filters: {'amenities': {'$in': ['gym']}, 'neighborhood': {'$in': ['financial-district']}, 'subway_routes': {'$in': ['a', 'c']}}
Combined Filters: [{'bedrooms': {'$eq': 2}, 'price': {'$lt': 5000.0}, 'amenities': {'$in': ['gym']}}, {'bedrooms': {'$eq': 2}, 'price': {'$lt': 5000.0}, 'neighborhood': {'$in': ['financial-district']}}, {'bedrooms': {'$eq': 2}, 'price': {'$lt': 5000.0}, 'subway_routes': {'$in': ['a', 'c']}}]


In [17]:
filter_list

[{'bedrooms': {'$eq': 2},
  'price': {'$lt': 5000.0},
  'amenities': {'$in': ['gym']}},
 {'bedrooms': {'$eq': 2},
  'price': {'$lt': 5000.0},
  'neighborhood': {'$in': ['financial-district']}},
 {'bedrooms': {'$eq': 2},
  'price': {'$lt': 5000.0},
  'subway_routes': {'$in': ['a', 'c']}}]

In [18]:
# Perform parallel hybrid search
raw_matches = parallel_hybrid_search(
    standalone_query, 
    filter_list,
    top_k=retrieval_k, 
    rerank=True
    )

[WARN] Filter {'bedrooms': {'$eq': 2}, 'price': {'$lt': 5000.0}, 'neighborhood': {'$in': ['financial-district']}} caused an error: list index out of range
[Score: 12.3029] ID: 4878034, $3700.0, 2.0BR/1.0BA, Neighborhoodlincoln-square, Boroughmanhattan
Amenities: ['concierge', 'courtyard', 'doorman', 'fios_available', 'garden_view', 'gym', 'hardwood_floors', 'laundry', 'live_in_super', 'package_room', 'virtual_doorman']
-----
[Score: 11.9506] ID: 4878050, $4295.0, 2.0BR/1.0BA, Neighborhoodmanhattan-valley, Boroughmanhattan
Amenities: ['city_view', 'dishwasher', 'fios_available', 'hardwood_floors', 'laundry', 'live_in_super', 'pets', 'private_roof_deck', 'roof_rights', 'skyline_view', 'smoke_free', 'washer_dryer']
-----
[Score: 11.8388] ID: 4878031, $3995.0, 2.0BR/1.0BA, Neighborhoodmanhattan-valley, Boroughmanhattan
Amenities: ['city_view', 'dishwasher', 'fios_available', 'hardwood_floors', 'laundry', 'pets', 'skyline_view', 'smoke_free', 'washer_dryer']
-----
[Score: 10.5356] ID: 48763

In [19]:
len(raw_matches)

16

In [20]:
from typing import Any, Dict, List, Tuple
def score_listing(match: Any, criteria: Dict[str, Any]) -> Tuple[float, List[str]]:
    """Compute a score and compromise list for a listing against user criteria.
    Heuristic scoring: 0..100.

    Criteria keys may include:
    - price: {"$lt": x} or {"$lte": x} or range (ignored for score if already filtered)
    - bedrooms: {"$eq": n}
    - bathrooms: {"$eq": n}
    - amenities: [..]
    - neighborhoods: [..]
    - subway: {"routes": [...], "lines": [...], "max_distance": float|None}
    """
    md = match.metadata
    score = 0.0
    compromises: List[str] = []

    # Bedrooms
    desired_bed = criteria.get("bedrooms", {}).get("$eq")
    if desired_bed is not None:
        if md.get("bedrooms") == desired_bed:
            score += 15
        else:
            compromises.append(f"bedrooms != {desired_bed}")

    # Bathrooms
    desired_bath = criteria.get("bathrooms", {}).get("$eq")
    if desired_bath is not None:
        if float(md.get("bathrooms", 0)) == float(desired_bath):
            score += 10
        else:
            compromises.append(f"bathrooms != {desired_bath}")

    # Price closeness (if budget exists)
    price_filter = criteria.get("price", {})
    budget = price_filter.get("$lt") or price_filter.get("$lte")
    if budget is not None:
        price = float(md.get("price", budget))
        if price <= budget:
            # Higher score if more under budget (cap at 15)
            under = max(0.0, (budget - price) / max(budget, 1))
            score += min(15.0, 15.0 * under * 2)
        else:
            compromises.append("over budget")

    # Neighborhood
    neighborhoods = criteria.get("neighborhoods") or []
    if neighborhoods:
        if (md.get("neighborhood", "").lower() in neighborhoods):
            score += 10
        else:
            compromises.append("different neighborhood")

    # Amenities (coverage)
    desired_amen = criteria.get("amenities") or []
    if desired_amen:
        listing_amen = [a.lower() for a in (md.get("amenities", []) or [])]
        covered = [a for a in desired_amen if a in listing_amen]
        coverage = len(covered) / max(1, len(desired_amen))
        score += 30.0 * coverage
        missing = [a for a in desired_amen if a not in listing_amen]
        if missing:
            compromises.append("missing amenities: " + ", ".join(missing))

    # Subway
    subway = criteria.get("subway") or {}
    routes = subway.get("routes") or []
    lines = subway.get("lines") or []
    max_dist = subway.get("max_distance")

    if routes or lines or (max_dist is not None):
        l_routes = [r.lower() for r in (md.get("subway_routes", []) or [])]
        l_lines = [l.lower() for l in (md.get("subway_lines", []) or [])]
        l_min = md.get("subway_min_distance")

        route_ok = (not routes) or any(r in l_routes for r in routes)
        line_ok = (not lines) or any(l in l_lines for l in lines)
        dist_ok = (max_dist is None) or (l_min is not None and l_min <= max_dist)

        # Determine how many conditions we’re checking
        route_weight = 1 if routes else 0
        line_weight = 1 if lines else 0
        dist_weight = 1 if max_dist is not None else 0

        score += 20.0 * (sum(1 for b in [route_ok, line_ok, dist_ok] if b) /
                        max(1, route_weight + line_weight + dist_weight))

        if not route_ok and routes:
            compromises.append("different subway route")
        if not line_ok and lines:
            compromises.append("different subway line")
        if not dist_ok and (max_dist is not None):
            compromises.append("farther from subway than desired")

    return round(score, 2), compromises


def score_listings(matches: List[Any], criteria: Dict[str, Any]) -> List[Tuple[Any, float, List[str]]]:
    """Return list of (match, score, compromises), sorted by score desc."""
    scored = [(* (m,),) + score_listing(m, criteria) for m in matches]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

In [21]:
# Step 4.5: Score listings against criteria for ordering and compromises
criteria = {
    "price": pinecone_filters.get("price") or {},
    "bedrooms": pinecone_filters.get("bedrooms") or {},
    "bathrooms": pinecone_filters.get("bathrooms") or {},
    "amenities": amenities or [],
    "neighborhoods": neighborhoods or [],
    "subway": subway_prefs or {},
}
scored = score_listings(raw_matches, criteria)
# Reorder by score
matches = [m for (m, _, _) in scored]

In [22]:
# Step 5: Trim to requested top_k after filtering
matches = matches[:top_k]

In [23]:
# Build clarification if results are sparse
clarification = None
if len(matches) < top_k // 2:
    missing = []
    if not pinecone_filters:
        missing.append("budget or bedroom count")
    if not neighborhoods and not amenities and not subway_prefs.get("routes"):
        missing.append("preferred neighborhood or subway line")
    if missing:
        clarification = f"I found {len(matches)} matches. For better results, could you specify: {', '.join(missing)}?"

In [24]:
# Step 6: Build current-turn retrieval context (include score/compromises inline)
# Create a quick map for compromises
id_to_score = {}
id_to_comp = {}
for m, s, comp in scored:
    mid = m.metadata.get("listing_id")
    id_to_score[mid] = s
    id_to_comp[mid] = comp

def format_with_scores(ms):
    lines = []
    for rank, m in enumerate(ms, start=1):
        md = m.metadata
        lid = md.get('listing_id')
        score = id_to_score.get(lid)
        retrieval_score = m.score
        comp = id_to_comp.get(lid) or []
        lines.append(
            f"{rank}. ID: {lid} (score: {score}; retrieval score: {retrieval_score})\n"
            f"   Price: ${md.get('price')}\n"
            f"   Bedrooms: {md.get('bedrooms')}, Bathrooms: {md.get('bathrooms')}\n"
            f"   Neighborhood: {md.get('neighborhood')}, Borough: {md.get('borough')}\n"
            f"   Amenities: {', '.join(md.get('amenities', []))}\n"
            f"   Compromises: {', '.join(comp) if comp else 'None'}\n"
        )
    return "\n".join(lines)

context_block = format_with_scores(matches)

In [25]:
# Step 7: Construct messages with conversation history
system_msg = {
    "role": "system",
    "content": (
        "You are RentIQ, a helpful assistant experienced in NYC rental markets. "
        "Use the conversation history to resolve pronouns and references like 'the first one' or 'the listing in Williamsburg'. "
        "Ground any recommendations strictly in the provided retrieved listings context for the current turn. "
        "If the user asks a follow-up that cannot be answered from context, ask a concise clarification question."
    ),
}

In [26]:
# Keep only the most recent turns to control token usage
history_messages = []
if chat_history:
    # Expecting a list of {role, content} items; filter to valid roles
    allowed_roles = {"user", "assistant", "system"}
    filtered = [m for m in chat_history if m.get("role") in allowed_roles and m.get("content")]
    # Trim to last 10 turns (approx 20 messages). Adjust as needed.
    history_messages = filtered[-10:]
print("History Messages:", history_messages)

History Messages: []


In [27]:
# Compose the current user prompt including retrieved context
if is_new_search:
    current_user_prompt = f"""
The user asked: "{user_query}"
Rewritten standalone query used for retrieval: "{standalone_query}"

Here are the top {top_k} listings retrieved from our database for this turn:
{context_block}

TASK:
- Provide a ranked list from most to least relevant.
- For each listing: summarize key selling points and match with the user's needs (from history and this turn).
- Be concise but informative.
- End with a brief final recommendation.
- If this is an UPDATED search (user changed requirements), acknowledge what changed.

Output only the recommendation list and summary.
"""
else:
    current_user_prompt = f"""
You are continuing an ongoing conversation. Answer naturally and concisely while grounding strictly in the retrieved listings for this turn.

User's latest message: "{user_query}"
Rewritten standalone query: "{standalone_query}"

Top {top_k} listings retrieved for this turn:
{context_block}

Guidelines:
- Keep a conversational tone. Avoid rigid ranking formatting unless explicitly requested.
- Reference prior preferences when relevant. If a referred listing is not in the current results, say so and suggest refining filters.
- Provide a succinct, helpful answer (2–5 sentences) and, when appropriate, suggest the next best question or adjustment.
"""

In [28]:
print(f"Current User Prompt:\n {current_user_prompt}")

Current User Prompt:
 
The user asked: "Looking for a 2 bedroom apartment under 5000, in fidi with a gym and close to subway lines A or C."
Rewritten standalone query used for retrieval: "Looking for a 2 bedroom apartment under 5000, in fidi with a gym and close to subway lines A or C."

Here are the top 5 listings retrieved from our database for this turn:
1. ID: 4878130 (score: 115.23; retrieval score: 8.72980309)
   Price: $3295.0
   Bedrooms: 2.0, Bathrooms: 1.0
   Neighborhood: washington-heights, Borough: manhattan
   Amenities: bike_room, city_view, courtyard, dishwasher, doorman, elevator, garage, gym, hardwood_floors, laundry, live_in_super, package_room, parking, recreation_facilities, storage_room
   Compromises: different neighborhood

2. ID: 4878034 (score: 112.8; retrieval score: 12.3028946)
   Price: $3700.0
   Bedrooms: 2.0, Bathrooms: 1.0
   Neighborhood: lincoln-square, Borough: manhattan
   Amenities: concierge, courtyard, doorman, fios_available, garden_view, gym, h

In [29]:
messages = [system_msg]
messages.extend(history_messages)
messages.append({"role": "user", "content": current_user_prompt})

In [30]:
# Step 8: Call LLM
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.2,
    max_tokens=1000
)
llm_output = response.choices[0].message.content.strip()

In [32]:
print(llm_output)

1. **ID: 4876461**
   - **Price:** $4300.0
   - **Neighborhood:** Fulton Seaport, Manhattan
   - **Key Selling Points:** This listing includes a gym, doorman, and is close to subway lines. It features hardwood floors, a roof deck, and is pet-friendly. 
   - **Match with User's Needs:** Meets the budget and bedroom requirement, and includes a gym.

2. **ID: 4876399**
   - **Price:** $4295.0
   - **Neighborhood:** South Harlem, Manhattan
   - **Key Selling Points:** Offers a gym, doorman, and laundry facilities. It also has a terrace and is pet-friendly.
   - **Match with User's Needs:** Fits the budget and bedroom requirement, but is in a different neighborhood.

3. **ID: 4877663**
   - **Price:** $4100.0
   - **Neighborhood:** Hamilton Heights, Manhattan
   - **Key Selling Points:** Features a gym, full-time doorman, and a roof deck. It is also furnished and pet-friendly.
   - **Match with User's Needs:** Meets the budget and bedroom requirement, but is in a different neighborhood.

4.

In [37]:
from mlx_lm import load, generate
from mlx_lm.models.cache import make_prompt_cache
import mlx.core as mx

In [34]:
# Load a 4-bit quantized Mistral 7B model 
model, tokenizer = load(
    "mlx-community/Mistral-7B-Instruct-v0.3-4bit",
    tokenizer_config={"trust_remote_code": True}
)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [36]:
# Load the model (I'm using a 3-bit Llama 3 here - even smaller!)
model, tokenizer = load("mlx-community/Mistral-7B-Instruct-v0.3-4bit")

# Create a memory-efficient prompt cache
prompt_cache = make_prompt_cache(
    model,
    max_kv_size=4096  # Limit cache size to prevent memory bloat
)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [46]:
messages = []

content = system_msg['content'] + "\n\n" + current_user_prompt

# Add user message to history
messages.append({"role": "user", "content": content})

# Format the conversation using the model's chat template
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

# Generate response using cached key-value pairs for efficiency
response = generate(
    model,
    tokenizer,
    prompt,
    prompt_cache=prompt_cache,
    kv_bits=4,             # Quantize cache to 4 bits
    kv_group_size=64,      # Quantization group size
    max_tokens=1000,
    temp=0.2,
)

[Warning] Specifying sampling arguments to ``generate_step`` is deprecated. Pass in a ``sampler`` instead.


In [47]:
print(response)

1. Recommendation: ID: 4878130 (Washington Heights)
   Key Selling Points:
   - 2 bedroom apartment under $4000
   - Gym and doorman amenities
   - Close to subway lines A and C
   - Hardwood floors, courtyard, bike room, laundry, storage room, and garage
   - Compromise: Different neighborhood from Fidi

2. Recommendation: ID: 4878034 (Lincoln Square)
   Key Selling Points:
   - 2 bedroom apartment under $4000
   - Gym and doorman amenities
   - Fios available
   - Hardwood floors, package room, virtual doorman, laundry, live-in super
   - Compromise: Different neighborhood from Fidi

3. Recommendation: ID: 4877663 (Hamilton Heights)
   Key Selling Points:
   - 2 bedroom apartment with 2 bathrooms under $4100
   - Gym, doorman, and furnished
   - Fios available, pets allowed, roof deck, skyline view
   - Compromise: Different neighborhood from Fidi

4. Recommendation: ID: 4876399 (South Harlem)
   Key Selling Points:
   - 2 bedroom apartment under $4300
   - Gym, doorman, and live-in 

In [48]:
# Clear GPU memory cache after large operations
mx.metal.clear_cache()